In [1]:
import sys
from pathlib import Path

from pyspark.sql import SparkSession

project_root = next(
    (
        candidate
        for candidate in (Path.cwd(), *Path.cwd().parents)
        if (candidate / "fraud_streaming" / "spark" / "src" / "data_processing.py").is_file()
    ),
    Path.cwd(),
)
spark_dir = project_root / "fraud_streaming" / "spark"
sys.path.insert(0, str(project_root))
sys.path.insert(0, str(spark_dir))

from src.data_processing import compute_window_analytics
from fraud_streaming.redis.config import (
    ANALYTICS_POLL_SECONDS,
    host as redis_host,
    port as redis_port,
)
from fraud_streaming.redis.redis_integration import RedisAnalyticsIntegration

spark = SparkSession.builder.appName("RedisFeatureWindowAnalysis").getOrCreate()
analytics_store = RedisAnalyticsIntegration(host=redis_host, port=redis_port)
analysis_checkpoint_path = (
    project_root / "data" / "checkpoints" / "feature_analysis"
)

In [2]:
def process_feature_analysis_batch(_timer_batch, batch_id):
    transactions = analytics_store.read_feature_store()
    metrics = compute_window_analytics(spark, transactions)
    analytics_store.write_analytics_to_redis(metrics)
    print(f"Updated Redis analytics for batch {batch_id}: {metrics}")

In [3]:
timer_stream = (
    spark.readStream.format("rate")
    .option("rowsPerSecond", 1)
    .load()
)

analysis_query = (
    timer_stream.writeStream
    .foreachBatch(process_feature_analysis_batch)
    .option("checkpointLocation", analysis_checkpoint_path.as_posix())
    .trigger(processingTime=f"{ANALYTICS_POLL_SECONDS} seconds")
    .start()
)
print(f"Started Redis feature analysis; polling every {ANALYTICS_POLL_SECONDS}s")

Started Redis feature analysis; polling every 5s


In [ ]:
try:
    analysis_query.awaitTermination()
finally:
    analysis_query.stop()
    analytics_store.close()

Updated Redis analytics for batch 283: {'transaction_count': 0, 'total_amount': 0.0, 'average_amount': 0.0, 'category_counts': {}, 'window_start': '2026-10-04T14:27:41+00:00', 'window_end': '2026-10-04T14:28:41+00:00', 'updated_at': '2026-10-04T14:28:41+00:00'}
Updated Redis analytics for batch 284: {'transaction_count': 0, 'total_amount': 0.0, 'average_amount': 0.0, 'category_counts': {}, 'window_start': '2026-10-04T14:28:05+00:00', 'window_end': '2026-10-04T14:29:05+00:00', 'updated_at': '2026-10-04T14:29:05+00:00'}
Updated Redis analytics for batch 285: {'transaction_count': 0, 'total_amount': 0.0, 'average_amount': 0.0, 'category_counts': {}, 'window_start': '2026-10-04T14:28:28+00:00', 'window_end': '2026-10-04T14:29:28+00:00', 'updated_at': '2026-10-04T14:29:28+00:00'}
Updated Redis analytics for batch 286: {'transaction_count': 0, 'total_amount': 0.0, 'average_amount': 0.0, 'category_counts': {}, 'window_start': '2026-10-04T14:28:51+00:00', 'window_end': '2026-10-04T14:29:51+00: